# Аналитика продаж кафе и доставки

Ноутбук загружает SQL-витрины из MS SQL Server, показывает базовые срезы и сохраняет графики в папку `reports/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'python' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'python'))

from load_data import load_data_marts

reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(exist_ok=True)


## Загрузка витрин

In [ ]:
data = load_data_marts()
{name: frame.shape for name, frame in data.items()}

## Продажи по дням

In [ ]:
daily_sales = data['daily_sales'].copy()
daily_sales['sales_date'] = pd.to_datetime(daily_sales['sales_date'])

ax = daily_sales.plot(x='sales_date', y='net_sales', marker='o', figsize=(10, 5), legend=False)
ax.set_title('Продажи по дням')
ax.set_xlabel('Дата')
ax.set_ylabel('Выручка')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(reports_dir / 'daily_sales.png', dpi=160)
plt.show()

## Товары и промокоды

In [ ]:
top_products = data['product_sales'].sort_values('gross_revenue', ascending=False).head(10)
ax = top_products.plot(kind='barh', x='product_name', y='gross_revenue', figsize=(10, 5), legend=False, color='#59A14F')
ax.set_title('Топ-10 товаров по выручке')
ax.set_xlabel('Валовая выручка')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(reports_dir / 'top_products.png', dpi=160)
plt.show()

promo = data['promo_effectiveness'].query('orders_with_promo > 0').sort_values('net_sales', ascending=False)
ax = promo.plot(kind='bar', x='promo_code', y='net_sales', figsize=(10, 5), legend=False, color='#F28E2B')
ax.set_title('Эффективность промокодов')
ax.set_xlabel('Промокод')
ax.set_ylabel('Выручка')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(reports_dir / 'promo_effectiveness.png', dpi=160)
plt.show()

## Статусы и платежи

In [ ]:
status_distribution = data['order_status_distribution']
ax = status_distribution.plot(kind='pie', y='orders_cnt', labels=status_distribution['order_status_ru'], autopct='%1.1f%%', figsize=(7, 7), legend=False)
ax.set_title('Распределение заказов по статусам')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(reports_dir / 'order_status_distribution.png', dpi=160)
plt.show()

payment_methods = data['payment_methods']
ax = payment_methods.plot(kind='barh', x='payment_method', y='payment_amount', figsize=(10, 5), legend=False, color='#76B7B2')
ax.set_title('Платежи по способам оплаты')
ax.set_xlabel('Сумма платежей')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(reports_dir / 'payment_methods.png', dpi=160)
plt.show()